In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import firebase_admin
from firebase_admin import credentials, db
import re
from datetime import datetime

class MedicalAnalysisSystem:
    def __init__(self, firebase_credentials_path):
        """Initialize the Medical Analysis System"""
        try:
            # Initialize Firebase if not already initialized
            if not firebase_admin._apps:
                cred = credentials.Certificate(firebase_credentials_path)
                firebase_admin.initialize_app(cred, {
                    'databaseURL': 'https://sarthak1proj-default-rtdb.firebaseio.com/'
                })

            self.MAX_RISK_SCORE = 10

            # Initialize dataset
            self.data = pd.DataFrame({
                'Symptom': ['Fever', 'Cough', 'Headache', 'Fatigue', 'Shortness of breath', 'Tiredness', 'Weakness'],
                'Condition': ['Flu', 'Cold', 'Migraine', 'Chronic Fatigue', 'Asthma', 'Liver Problem', 'General Weakness'],
                'Description': ['High body temperature', 'Dry or wet cough', 'Severe headache', 'Persistent tiredness',
                              'Difficulty breathing', 'Persistent tiredness', 'General body weakness'],
                'Risk_Score': ['Low', 'Medium', 'High', 'Medium', 'High', 'High', 'Medium']
            })

            # Set up risk mapping
            risk_map = {"Low": 1, "Medium": 2, "High": 3}
            self.data = self.data.assign(
                Risk_Score_Encoded=self.data['Risk_Score'].map(risk_map).fillna(0)
            )

        except Exception as e:
            raise Exception(f"Initialization error: {str(e)}")

    def extract_info_from_bot_response(self, bot_response_data):
        """Extract information from bot response"""
        try:
            if not bot_response_data:
                return 0, [], {}

            bot_response_text = str(bot_response_data) if isinstance(bot_response_data, (dict, list)) else bot_response_data

            # Extract age
            age_match = re.search(r'(\d+)[\s-]years?[\s-]old', bot_response_text.lower())
            age = int(age_match.group(1)) if age_match else 0

            # Extract symptoms
            symptoms = []
            bot_response_lower = bot_response_text.lower()
            for symptom in self.data['Symptom']:
                if symptom.lower() in bot_response_lower:
                    symptoms.append(symptom)

            # Extract symptom intervals
            symptom_intervals = {}
            interval_keywords = {
                'daily': 'daily',
                'weekly': 'weekly',
                'monthly': 'monthly',
                'occasionally': 'occasional'
            }

            for symptom in symptoms:
                interval_found = False
                for keyword, value in interval_keywords.items():
                    if keyword in bot_response_lower:
                        symptom_intervals[symptom] = value
                        interval_found = True
                        break
                if not interval_found:
                    symptom_intervals[symptom] = 'unknown'

            return age, symptoms, symptom_intervals

        except Exception as e:
            print(f"Error in extract_info_from_bot_response: {str(e)}")
            return 0, [], {}

    def calculate_risk_score(self, age, symptoms):
        """Calculate risk score and percentage with higher emphasis on symptoms and reduced age impact"""
        try:
            matched_data = self.data[self.data['Symptom'].isin(symptoms)]
            symptom_risk_score = matched_data['Risk_Score_Encoded'].sum() * 2
            age_risk = max(0, (age // 10) * 0.5)
            final_risk_score = min(symptom_risk_score + age_risk, self.MAX_RISK_SCORE)
            risk_percentage = (final_risk_score / self.MAX_RISK_SCORE) * 100
            risk_percentage = min(100, max(0, risk_percentage))

            if risk_percentage <= 30:
                risk_label = "Low"
            elif risk_percentage <= 70:
                risk_label = "Medium"
            else:
                risk_label = "High"

            return risk_label, final_risk_score, risk_percentage

        except Exception as e:
            print(f"Error in calculate_risk_score: {str(e)}")
            return "Low", 0, 0

    def parse_timestamp(self, timestamp_str):
        """Convert ISO timestamp string to Unix timestamp, handling errors gracefully."""
        try:
            if isinstance(timestamp_str, str) and timestamp_str not in ['0', '', None]:
                dt = datetime.strptime(timestamp_str, '%Y-%m-%dT%H:%M:%S.%fZ')
                return dt.timestamp()
            return 0
        except Exception as e:
            print(f"Warning: Invalid timestamp format: {timestamp_str} - {str(e)}")
            return 0

    def process_user_data(self):
        """Fetch the latest user message from Firebase and analyze it."""
        try:
            ref = db.reference('/chats/user_123/')
            all_responses = ref.get()

            if not all_responses:
                return {"error": "No data found in Firebase"}

            messages = []

            if isinstance(all_responses, dict):
                for key, value in all_responses.items():
                    if isinstance(value, dict):
                        timestamp = self.parse_timestamp(value.get('timestamp', '0'))
                        messages.append((timestamp, value))
                    else:
                        messages.append((0, value))

                messages.sort(key=lambda x: x[0], reverse=True)
                latest_message = messages[0][1]

                if isinstance(latest_message, dict) and 'message' in latest_message:
                    latest_message = latest_message['message']
            else:
                latest_message = all_responses[-1]

            age, symptoms, symptom_intervals = self.extract_info_from_bot_response(latest_message)

            if not symptoms:
                return {"error": "No symptoms detected in the response"}

            risk_label, risk_score, risk_percentage = self.calculate_risk_score(age, symptoms)

            possible_conditions = self.data[self.data['Symptom'].isin(symptoms)]['Condition'].unique().tolist()

            analysis_results = {
                'age': age,
                'symptoms': symptoms,
                'symptom_intervals': symptom_intervals,
                'risk_label': risk_label,
                'risk_score': risk_score,
                'risk_percentage': round(risk_percentage, 1),
                'possible_conditions': possible_conditions,
                'analysis_timestamp': {'.sv': 'timestamp'},
                'analyzed_message': latest_message
            }

            self.update_firebase('/chats/user_123/analysis', analysis_results)

            return analysis_results

        except Exception as e:
            return {"error": f"Error processing data: {str(e)}"}

    def update_firebase(self, path, data):
        """Update Firebase with analysis results"""
        try:
            ref = db.reference(path)
            data['updated_at'] = {'.sv': 'timestamp'}
            ref.set(data)
            return True
        except Exception as e:
            print(f"Error updating Firebase: {str(e)}")
            return False

    def print_analysis_results(self, results):
        """Print analysis results including the message that was analyzed"""
        if "error" not in results:
            print("\nAnalysis Results:")
            print("-" * 50)
            print(f"Analyzed Message:\n{results.get('analyzed_message', 'Message not available')}")
            print(f"\nAge: {results['age']} years")
            print(f"Detected Symptoms: {', '.join(results['symptoms'])}")
            print(f"Risk Level: {results['risk_label']}, Risk Score: {results['risk_score']}/10, Risk Percentage: {results['risk_percentage']}%")
            print("Possible Conditions:", ", ".join(results['possible_conditions']))
        else:
            print(f"Error: {results['error']}")

def main():
    system = MedicalAnalysisSystem("firebase_credentials.json")
    results = system.process_user_data()
    system.print_analysis_results(results)

if __name__ == "__main__":
    main()



Analysis Results:
--------------------------------------------------
Analyzed Message:
{'bot_response': 'Your symptoms suggest mild to moderate concern. Keep tracking your condition and seek medical help if it worsens.', 'timestamp': '2025-02-17T16:21:29.204Z', 'user_message': '40 year old man with cough and cold for 2 days , with a headache during overworking , heart rate 84bpm'}

Age: 40 years
Detected Symptoms: Cough, Headache
Risk Level: High, Risk Score: 10/10, Risk Percentage: 100%
Possible Conditions: Cold, Migraine
